In [14]:
import numpy as np
import os
import time
import yaml

import meshcat
import meshcat.geometry as g
import meshcat.transformations as tf

from tempfile import TemporaryDirectory

from PIL import Image
import io
from math_utils import transform_bundletrack_output
# video resolution
video_resolution=[640, 480]

#
## parameters:
#
toss_id=1
cam = 'cam0'
video_name = f'cube_hand_{toss_id}'
cam_poses_file = './assets/realsense_pose_cube_hand_60_2.yaml'
output_file = './' + video_name + '.mp4'
yaml_path = './assets/config.yaml'
toss_type = 'cube'

intrinsics = [380.2484436035156, 379.8265380859375,314.2138977050781, 240.59800720214844,] # fx, fy, cx, cy

with open(cam_poses_file, 'r') as stream:
    data_loaded = yaml.safe_load(stream)
print(data_loaded[cam]['pose']['position']['x'])

cam_pos_dict = data_loaded[cam]['pose']['position']
cam_position = [cam_pos_dict['x'], cam_pos_dict['y'], cam_pos_dict['z']]
cam_rot_dict = data_loaded[cam]['pose']['rotation']
cam_orientation = [cam_rot_dict['x'], cam_rot_dict['y'], cam_rot_dict['z']]

# For extrinsic matrix
cam_trans = np.array([cam_pos_dict['x'], cam_pos_dict['y'], cam_pos_dict['z']]).reshape(-1, 1)
cam_axis_vec = np.array([cam_rot_dict['x'], cam_rot_dict['y'], cam_rot_dict['z']])

# Compute T_WC, transform from world to camera
cam_angle = np.linalg.norm(cam_orientation)
cam_axis = cam_orientation/cam_angle
T_WC = tf.translation_matrix(cam_position) @ tf.quaternion_matrix(tf.quaternion_about_axis(cam_angle, cam_axis))
print(tf.quaternion_matrix(tf.quaternion_about_axis(cam_angle, cam_axis)))

# Given fov_x, fov_y, desire output to be h x w resolution
# set fov=fov_y, screen_h = h
# set screen_w = fov_x/fov_y * w

fx = intrinsics[0]
fy = intrinsics[1]

fov_x = 2*np.arctan(video_resolution[1]/(2*fy))*180/np.pi
fov_y = 2*np.arctan(video_resolution[0]/(2*fx))*180/np.pi


cam_fov = fov_x

print('Extracted field of view: ' + f'{cam_fov:f}')


resolution = video_resolution

vis = meshcat.Visualizer()

1.12044811
[[ 0.5041678   0.53538885 -0.6776235   0.        ]
 [ 0.86198486 -0.2639196   0.43281467  0.        ]
 [ 0.05288603 -0.80231242 -0.59455685  0.        ]
 [ 0.          0.          0.          1.        ]]
Extracted field of view: 64.574913
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7002/static/


In [15]:
# Create the cubes. Set the opacity of the "real" to > 0 if you want to see it, for invisible
vis["real_1"].set_object(g.Box([0.1048, 0.1048, 0.1048]),
                       g.MeshLambertMaterial(
                             color=0x00ff00,
                             reflectivity=0.0,
                             transparent=0,
                             opacity=.4))

# vis["real_1"].set_object(g.Box([0.096, 0.061, 0.096]),
#                        g.MeshLambertMaterial(
#                              color=0x00ff00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))

In [17]:
from math_utils import trans_mat_to_pos_quat


def extract_times(messages):
    t_ros = np.zeros(len(messages))
    for i, data in enumerate(list(messages)):
        (_, msg, _) = data
        tstamp = msg.header.stamp
        t_ros[i] = tstamp.secs + tstamp.nsecs * 1e-9

    return t_ros


def extract_poses(messages, start_time, end_time):
    # poses = np.zeros((7, len(messages)))
    poses = np.zeros((7,1))
    for i, data in enumerate(messages):
        (_, msg, _) = data
        if start_time <= msg.header.stamp < end_time:
            pose = msg.pose.pose
            pose_pos = np.asarray([pose.position.x, pose.position.y, pose.position.z])
            pose_quat = np.asarray([pose.orientation.w, pose.orientation.x, pose.orientation.y,
                                    pose.orientation.z])
            # print(np.hstack((pose_quat, pose_pos)).T.shape)
            poses = np.hstack((poses, np.hstack((pose_quat, pose_pos)).T.reshape((-1,1))))
            # poses[:4, i] = pose_quat
            # poses[4:7, i] = pose_pos
    return poses[:,1:]

def get_bundletrack_results():
    """
    State vector is 4 quaternion + 3 xyz position + 3 angular velocity + 3 linear velocity.
    """
    frame_num = len([name for name in os.listdir(DATA_DIR)])
    print("%i frames in total!"%frame_num)
    poses = np.zeros((7, frame_num))
    for frame_id in range(1, frame_num+1):
        pose = np.loadtxt(DATA_DIR + "%04i.txt" % frame_id)#in camera frame
        pose = transform_bundletrack_output(
            pose,
            DATA_DIR,
            ODOM_FILE_PATH,
            cam_trans,
            cam_axis_vec,
            to_world=True
        )
        pose_quat = trans_mat_to_pos_quat(pose)[3:]
        pose_quat = pose_quat.reshape(1, -1)
        pose_pos = np.array(pose[:3, 3])
        poses[:4, frame_id-1] = pose_quat
        poses[4:7, frame_id-1] = pose_pos
    return poses

# DATA_DIR = os.path.join(os.getcwd(), "ob_in_cams", "ob_in_cam_cube_hand/")
DATA_DIR = os.path.join(os.getcwd(), "..", "BundleSDF", "results", video_name, "ob_in_cam/")
BUNDLESDF_DATA = os.path.join(os.getcwd(), "..", "BundleSDF", "data")
ODOM_FILE_PATH = os.path.join(BUNDLESDF_DATA, video_name, "annotated_poses/")
bundletrack_poses = get_bundletrack_results()

552 frames in total!


In [18]:
base_url = "http://127.0.0.1"

meshcat_url = base_url + ":" + vis.url().split(":")[-1]

''' Create precisely-sized iframe with meshcat view; put this in its own Jupyter cell. '''
from IPython.display import HTML
frame_html = """
<div style="height: {height}px; width: {width}px; overflow-x: visible; overflow-y: visible; resize: none">
    <iframe src="{url}" style="width: 100%; height: 100%; border: none"></iframe>
</div>
""".format(url=meshcat_url, width=resolution[0], height=resolution[1])
HTML(frame_html)

In [19]:
# Frames
# (W) World
# (M) Meshcat
# (C) Camera
# (A) link_1
# (B) link_2

vis["cam"].set_transform(T_WC)
vis["cam_view"].set_transform(T_WC @ tf.translation_matrix([0,0,.05]))

# T_MC, look along z-axis but rotote by 180 degrees
T_MC = tf.translation_matrix([0, 0, -1]) @ tf.rotation_matrix(np.pi, (0,0,1))

T_MW = T_MC @ tf.inverse_matrix(T_WC)

cam = g.PerspectiveCamera(fov=cam_fov, zoom=1, aspect=640/480)
vis["/Cameras/default/rotated"].set_object(cam)

# vis["/Cameras/default/rotated/<object>"].set_property("zoom", 1)
vis["/Cameras/default/rotated/<object>"].set_property("position", [0,0,0])
vis["/Cameras/default"].set_transform(T_MC)



In [20]:
# view in meshcat save to images
# Turn off background, axes, and grid.
vis['/Background'].set_property("visible", False)
vis['/Grid'].set_property("visible", False)
vis['/Axes'].set_property("visible", False)
with TemporaryDirectory(prefix="ros-process-") as tmpdir:
    print(tmpdir)
    for i, pose in enumerate(bundletrack_poses.T):
        print('Processing frame ' + f'{i:d}' + ' of ' + f'{bundletrack_poses.shape[1]:d}', end='\r')
        print(i)
        # im = Image.open(io.BytesIO(cam_data[i])).convert('RGB')
        rgb_file_name = os.path.join(BUNDLESDF_DATA, "cube_hand_1", "rgb", f"{i+1:04}.png")
        im = Image.open(rgb_file_name).convert('RGB')
        # im = Image.fromarray(cam_data[i]).convert('RGB')
        T_WA = tf.translation_matrix(pose[4:7]) @ tf.quaternion_matrix(pose[:4])
        vis["real_1"].set_transform(T_MW @ T_WA)

        # pose_2 = elbow_1[:,i]
        # T_WB = tf.translation_matrix(pose_2[4:7]) @ tf.quaternion_matrix(pose_2[:4])
        # vis["real_2"].set_transform(T_MW @ T_WB)

        mesh_im = vis.get_image()
        # print(mesh_im.size)
        # mesh_im.show()
        im.paste(mesh_im, (0,0), mask = mesh_im)
        # im.show()
        # break
        im.save(tmpdir + '/' + f'{i:07d}' + '.png', format="png")
    os.system('ffmpeg -y -r 150 -i ' + tmpdir + '/%07d.png -vcodec libx264 -preset slow -crf 18 ' + output_file)

/tmp/ros-process-txme_lxa
0rocessing frame 0 of 552
1rocessing frame 1 of 552
2rocessing frame 2 of 552
3rocessing frame 3 of 552
4rocessing frame 4 of 552
5rocessing frame 5 of 552
6rocessing frame 6 of 552
7rocessing frame 7 of 552
8rocessing frame 8 of 552
9rocessing frame 9 of 552
10ocessing frame 10 of 552
11ocessing frame 11 of 552
12ocessing frame 12 of 552
13ocessing frame 13 of 552
14ocessing frame 14 of 552
15ocessing frame 15 of 552
16ocessing frame 16 of 552
17ocessing frame 17 of 552
18ocessing frame 18 of 552
19ocessing frame 19 of 552
20ocessing frame 20 of 552
21ocessing frame 21 of 552
22ocessing frame 22 of 552
23ocessing frame 23 of 552
24ocessing frame 24 of 552
25ocessing frame 25 of 552
26ocessing frame 26 of 552
27ocessing frame 27 of 552
28ocessing frame 28 of 552
29ocessing frame 29 of 552
30ocessing frame 30 of 552
31ocessing frame 31 of 552
32ocessing frame 32 of 552
33ocessing frame 33 of 552
34ocessing frame 34 of 552
35ocessing frame 35 of 552
36ocessing f

ffmpeg version 4.2.7-0ubuntu0.1 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 9 (Ubuntu 9.4.0-1ubuntu1~20.04.1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-avresample --disable-filter=resample --enable-avisynth --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librsvg --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --e

In [6]:
# # view in meshcat save to images
# # Turn off background, axes, and grid.
# vis['/Background'].set_property("visible", False)
# vis['/Grid'].set_property("visible", False)
# vis['/Axes'].set_property("visible", False)
# import cv2
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter('overlay_video.mp4', fourcc, 20.0, (640, 480))
# # with TemporaryDirectory(prefix="ros-process-") as tmpdir:
#     # print(tmpdir)
# for i, pose in enumerate(bundletrack_poses.T):
#     print('Processing frame ' + f'{i:d}' + ' of ' + f'{bundletrack_poses.shape[1]:d}', end='\r')
#     print(i)
#     # im = Image.open(io.BytesIO(cam_data[i])).convert('RGB')
#     rgb_file_name = os.path.join(BUNDLESDF_DATA, "cube_hand_toss_60_2", "rgb", f"{i+1:04}.png")
#     # im = cv2.imread(rgb_file_name)
#     im = Image.open(rgb_file_name)
#     # im = Image.fromarray(cam_data[i]).convert('RGB')
#     T_WA = tf.translation_matrix(pose[4:7]) @ tf.quaternion_matrix(pose[:4])
#     vis["real_1"].set_transform(T_MW @ T_WA)
#     # pose_2 = elbow_1[:,i]
#     # T_WB = tf.translation_matrix(pose_2[4:7]) @ tf.quaternion_matrix(pose_2[:4])
#     # vis["real_2"].set_transform(T_MW @ T_WB)

#     mesh_im = vis.get_image()
#     im.paste(mesh_im, (0,0), mask = mesh_im)
#     # im.show()
#     # break
#     # im.save(tmpdir + '/' + f'{i:07d}' + '.png', format="png")
#     im_np = np.array(im)
#     im_np = cv2.cvtColor(im_np, cv2.COLOR_RGB2BGR)
#     out.write(im_np)
# out.release()
# # os.system('ffmpeg -y -r 150 -i ' + tmpdir + '/%07d.png -vcodec libx264 -preset slow -crf 18 ' + output_file)

0rocessing frame 0 of 780
1rocessing frame 1 of 780
2rocessing frame 2 of 780
3rocessing frame 3 of 780
4rocessing frame 4 of 780
5rocessing frame 5 of 780
6rocessing frame 6 of 780
7rocessing frame 7 of 780
8rocessing frame 8 of 780
9rocessing frame 9 of 780
10ocessing frame 10 of 780
11ocessing frame 11 of 780
12ocessing frame 12 of 780
13ocessing frame 13 of 780
14ocessing frame 14 of 780
15ocessing frame 15 of 780
16ocessing frame 16 of 780
17ocessing frame 17 of 780
18ocessing frame 18 of 780
19ocessing frame 19 of 780
20ocessing frame 20 of 780
21ocessing frame 21 of 780
22ocessing frame 22 of 780
23ocessing frame 23 of 780
24ocessing frame 24 of 780
25ocessing frame 25 of 780
26ocessing frame 26 of 780
27ocessing frame 27 of 780
28ocessing frame 28 of 780
29ocessing frame 29 of 780
30ocessing frame 30 of 780
31ocessing frame 31 of 780
32ocessing frame 32 of 780
33ocessing frame 33 of 780
34ocessing frame 34 of 780
35ocessing frame 35 of 780
36ocessing frame 36 of 780
37ocessing 